In [ ]:
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import subprocess
import os
import utm
import xarray as xr
import rasterio as rio
from pyproj import Transformer
import sys
import glob
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor
import warnings
warnings.filterwarnings('ignore')

### Inputs

In the following cell, we ask for a few basic inputs from you, and use standardized values for physical & rheological properties in each of our models. For the time dependant models (IMEX_Lava & LAVA2D), we ask for a runtime value, as well as a variable to downscale the resolution by to speed up the simulations.

If you would like to further edit the models, refer further down to additional guidance unique to each simulation.

In [ ]:
dem = "/home/jovyan/shared/DEMs/Iceland_20mx20m_cropped.tif"

vent_latitude = 63.890688
vent_longitude = -22.269106

volume = 1e8

name = "fagradalsfjall"

# IMEX and Lava2d are depth-averaged time-dependent models, and thus require an approximate run time.
# Below, we assign a reasonable yet arbitrary time for the flow to run (in seconds)
# Please change it at your leisure
run_time = 1000

# Please enter 1, 2, or 4 (original resolution, half, quarter respectively)
downscale = 2

In [ ]:
x = rio.open(dem)
bounds = x.bounds
crs = x.crs

transformer = Transformer.from_crs(crs.to_string(), "EPSG:4326")
transformer2 = Transformer.from_crs("EPSG:4326", crs.to_string())
converted = transformer2.transform(vent_latitude,vent_longitude)

vent_easting = converted[0]
vent_northing = converted[1]

In [ ]:
vent_easting

In [ ]:
victor.plot_dem(dem, np.array([vent_easting,vent_northing]))

In [ ]:
raster = rxr.open_rasterio(dem)
raster = raster.sel({"band": 1})
xll, yll = int(raster.x.min()), int(raster.y.min())
coordinates = np.array([int(vent_easting),int(vent_northing)])
r = rio.open(dem)
res = r.res

output_num = str(int(run_time/100))
out = output_num.zfill(4)

filename, file_extension = os.path.splitext(dem)
file_name = os.path.basename(dem)
file_name = os.path.splitext(file_name)[0]
type_flag = 0 #0 is ascii, 1 is tiff
if file_extension in [".geotiff", ".tiff", ".tif"]:
    type_flag = 1
    asc_name = "".join((file_name, ".asc"))
    raster.rio.to_raster(asc_name)
    os.remove("".join((asc_name, ".aux.xml")))
    # os.remove("".join((file_name, ".prj")))
    cell_size = open(f"""{asc_name}""","r+")
    info = cell_size.readlines()
    uniform = info[4].split()
    if uniform[0] == 'dx':
        info[4] = f"""cellsize     {int(float(uniform[1]))}\n"""
        info[5] = ""
        cell_size.seek(0)
        cell_size.writelines(info)
        cell_size.truncate()
        cell_size.close()
elif file_extension in [".asc", ".ascii"]:
    type_flag = 0
    tiff_name = "".join((file_name, ".tiff"))
    raster.rio.to_raster(tiff_name)

### Molasses: Additional Notes

Molasses input parameters are stored in `custom_molasses.conf` in the `Molasses` folder. The most impactful change would be that of changing the minimum and maximum residual, as the cellular automata structure will cause a much larger or smaller area to be impacted. As a stochastic model, no physical parameters are used.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/Molasses")
f = open("custom_molasses.conf","r+")
conf = f.readlines()
conf[2] = "MIN_RESIDUAL = 1\n"
conf[3] = "MAX_RESIDUAL = 20\n"
conf[4] = f"""MIN_TOTAL_VOLUME = {str(volume)}\n"""
conf[5] = f"""MAX_TOTAL_VOLUME = {str(volume)}\n"""
conf[6] = f"""MIN_PULSE_VOLUME = {float(volume)/100}\n"""
conf[7] = f"""MAX_PULSE_VOLUME = {float(volume)/100}\n"""
conf[10] = f"""DEM_FILE = {dem}\n"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

f=open("events.in","w")
f.write(f"""{vent_easting},{vent_northing}""")
f.close()

if not os.path.exists(f"../{dem}"):
    subprocess.run(f"""ln -s ../{dem} .""",shell=True)
subprocess.run("molasses custom_molasses.conf",shell=True,stderr=subprocess.DEVNULL)

victor.convert_molasses(f"molasses_{name}")
os.chdir("/home/jovyan/Lava Benchmark")

In [ ]:
victor.plot_flow(dem,f"./Molasses/molasses_{name}.asc")

### MrLavaLoba: Additional Notes

MrLavaLoba input parameters are stored in `input_data.py` in the `mr_lava_loba` folder. The most impactful changes would be those such as the thickening parameter, lobe exponent, or max slope probability. Other parameters not mentioned will also affect the creation and placement of lobes. As a stochastic model, no physical parameters are specified by the user.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/mr_lava_loba")
g=open("input_data.py","r+")
inp = g.readlines()
inp[1] = f"""run_name = '{name}'\n"""
if type_flag == 1:
    inp[3] = f"""source = '{asc_name}'\n"""
else:
    inp[3] = f"""source = '{dem}'\n"""
inp[41] = f"""vent_flag = 0\n"""
inp[44] = f"""x_vent = [{int(vent_easting)}]\n"""
inp[45] = f"""y_vent = [{int(vent_northing)}]\n"""
inp[91] = f"""total_volume = {volume}\n"""
inp[96] = f"""fixed_dimension_flag = 1\n"""
g.seek(0)
g.writelines(inp)
g.truncate()
g.close()

if type_flag == 1:
    subprocess.run(f"""ln -s ../{asc_name} .""",shell=True)
else:
    subprocess.run(f"""ln -s ../{dem} .""",shell=True)
subprocess.run("python mr_lava_loba.py",shell=True)
if type_flag == 1:
    os.remove(asc_name)
else:
    os.remove(dem)
os.chdir("/home/jovyan/Lava Benchmark/")

In [ ]:
mll_outputs = glob.glob(f'mr_lava_loba/{name}*thickness_masked*.asc')
mll_out = mll_outputs[-1]
victor.plot_flow(dem,mll_out)

### IMEX_Lava: Additional Notes

IMEX_Lava input parameters are stored in `IMEX_LavaFlow.inp` in the `IMEX` folder. This is a physics-based, highly accurate, slower model. If desiring to change additional input parameters, we recommend editing the `TEMPERATURE_PARAMETERS`, `RHEOLOGY_PARAMETERS`, `GAS_TRANSPORT_PARAMETERS`, and `LIQUID_TRANSPPORT_PARAMETERS` namelists as needed. All other sections are carefully prepared based in your initial basic inputs, so ignore all other sections.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/IMEX_LavaFlow")
array = rxr.open_rasterio(dem)
h=open("IMEX_LavaFlow.inp","r+")
inp = h.readlines()
inp[1] = f''' RUN_NAME="{name}"\n'''
inp[4] = f""" T_END=  {run_time},\n"""
inp[5] = f""" DT_OUTPUT=  100,\n"""
inp[13] = f""" X0= {int(xll)+1}     ,\n"""
inp[14] = f""" Y0= {int(yll)+1}     ,\n"""
inp[15] = f""" COMP_CELLS_X=  {int(len(array.x)/downscale)-5},\n"""
inp[16] = f""" COMP_CELLS_Y=  {int(len(array.y)/downscale)-5},\n"""
inp[17] = f""" CELL_SIZE = {res[0]*downscale}D0 ,\n"""
inp[103] = f"""  X_SOURCE=  {int(vent_easting)}.0D0 ,\n"""
inp[104] = f"""  Y_SOURCE=  {int(vent_northing)}.0D0 ,\n"""
inp[105] = f"""  R_SOURCE=  {res[0]}D0     ,\n"""
inp[106] = f""" VEL_SOURCE= {float(volume)/(res[0]**2*np.pi*run_time)} \n"""
h.seek(0)
h.writelines(inp)
h.truncate()
h.close()

os.remove("topography_dem.asc")
if type_flag == 1:
    subprocess.run(f"""ln -s ../{asc_name} topography_dem.asc""",shell=True)
else:
    subprocess.run(f"""ln -s {dem} topography_dem.asc""",shell=True)
subprocess.run("./IMEX_LavaFlow",shell=True)
os.chdir("/home/jovyan/Lava Benchmark")

In [ ]:
victor.plot_flow(dem, f"""./IMEX/{name}_{out}.asc""")

### LAVA2D: Additional Notes

LAVA2D input parameters are stored in `input.py` in the `lava2d` folder. This is a physics-based, highly accurate, slower model. If desiring to change additional input parameters, we recommend editing the `set_vent_props`, `set_lava_props`, `set_rheo`, `set_ambient`, and `set_numeric` sections as needed. All other sections are carefully prepared based in your initial basic inputs, so please leave them as is.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/lava2d")
rast = rio.open(dem)
left, bottom, right, top = rast.bounds
lower_left = transformer.transform(left, bottom)
top_right = transformer.transform(right, top)
src = transformer.transform(vent_easting, vent_northing)

l = open("input.py","r+")
inp = l.readlines()

if type_flag == 0:
    inp[4] = f"""    path_to_dem_file    = ('{tiff_name}'),\n"""
else:
    inp[4] = f"""    path_to_dem_file    = ('{dem}'),\n"""
inp[5] = f"""    Lon_SRC             = {src[1]}, # source longitude\n"""
inp[6] = f"""    Lat_SRC             = {src[0]},    # source latitude\n"""
inp[7] = f"""    Lon_LowerLeft       = {lower_left[1]}, # bounding box: lower-left longitude\n"""
inp[8] = f"""    Lat_LowerLeft       = {lower_left[0]}, # bounding box: lower-left latitude\n"""
inp[9] = f"""    Lon_UpperRight      = {top_right[1]}, # bounding box: upper-right longitude\n"""
inp[10] = f"""    Lat_UpperRight      = {top_right[0]},   # bounding box: upper-right latitude\n"""
inp[12] = f"""    dx_desired          = {int(res[0]*downscale)},          # meters\n"""
inp[67] = f"""    max_iter = None, #one of them can be none, so the default value for both is none, if both none run till killed\n"""
inp[68] = f"""    max_time_hr = {run_time/3600},\n"""
l.seek(0)
l.writelines(inp)
l.truncate()
l.close()

k=open("./example_vents/vent_01.txt","r+")
inp = k.readlines()
inp[1] = f'''0	50	0	-50	0	50   10000\n'''
inp[2] = f'''43200	0	0	100	-75	20    10000\n'''
inp[3] = f'''54000	0	0	100	-75	20    5000\n'''
inp[4] = f'''86400	0	0	100	-75	20    5000\n'''

k.seek(0)
k.writelines(inp)
k.truncate()
k.close()

subprocess.run("python input.py",shell=True)

In [ ]:
data = xr.open_dataset("outputs/out.nc",group="DATA/PHYSICS",decode_coords="all")
data = data.where(data!=0, np.nan)

coord = xr.open_dataset("outputs/out.nc",group="DATA", decode_coords="all")
lat = coord.lat[:,0]
lon = coord.lon[0,:]
data['rows'] = lat.values
data['cols'] = lon.values

data = data.rename({"rows": "y", "cols": "x"})
repeated_arr_x = np.repeat(data.y.values[0], len(data.x))
repeated_arr_y = np.repeat(data.x.values[0], len(data.y))

utm1 = transformer2.transform(repeated_arr_x,data.x.values)
utm2 = transformer2.transform(data.y.values,repeated_arr_y)
# print(utm1[0], utm2[1])
data['x'] = utm1[0]
data['y'] = utm2[1]
data.lava_thickness_total.rio.to_raster(f"{name}_output.tif")

In [ ]:
victor.plot_flow(dem, f"{name}_output.tif")

## Visualizing the results

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/")

r = rxr.open_rasterio(f"IMEX_LavaFlow/{name}_max.asc",masked=True)
r2 = rxr.open_rasterio(f"Molasses/molasses_{name}.asc",masked=True)
r3 = rxr.open_rasterio(mll_out,masked=True)
r4 = rxr.open_rasterio(f"lava2d/{name}_output.tif",masked=True)
r_min, r_max = r.min(), r.max()
r2_min, r2_max = r2.min(), r2.max()
r3_min, r3_max = r3.min(), r3.max()
r4_min, r4_max = r4.min(), r4.max()

maximum = np.max((r_max,r2_max,r3_max,r4_max))
idx_max = np.argmax((r_max,r2_max,r3_max,r4_max))

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2, figsize = (12,9))

thickness0 = victor.plot_flow(dem, f"Molasses/molasses_{name}.asc", axes=ax0,zoom=False,title='Molasses',minimum=1e-6,scale=maximum)
thickness1 = victor.plot_flow(dem, mll_out, axes=ax1, zoom=False,title='MrLavaLoba/Flowy',minimum=1e-6, scale=maximum)
thickness2 = victor.plot_flow(dem, f"IMEX_LavaFlow/{name}_max.asc", axes=ax2, zoom=False,title='IMEX_LavaFlow', minimum=1e-6, scale=maximum)
thickness3 = victor.plot_flow(dem, f"lava2d/{name}_output.tif", axes=ax3 ,title='Lava2D', minimum=1e-10, scale=maximum*100)


cbar_ax = fig.add_axes([.25, 0, 0.5, 0.05])
thickness = [thickness0, thickness1, thickness2, thickness3]
thickness = thickness[idx_max]
fig.colorbar(thickness, cax=cbar_ax, label="Flow Thickness (m)", orientation="horizontal")